# Data Cleaning — The Host Premium: Quantifying Home Advantage in the FIFA World Cup Era

This notebook prepares **five clean CSV files** for use in `proposal_viz.ipynb`:

| # | Output file | Source | Purpose |
|---|-------------|--------|---------|
| 1 | `clean/wc_matches.csv` | Match_Results.csv | All 964 WC matches with host flags & goal diffs |
| 2 | `clean/baseline_rates.csv` | Match_Results.csv | Home/neutral/WC-host win rates across all intl. matches |
| 3 | `clean/host_elo_delta.csv` | soccer-elo.csv | Pre-tournament Elo rank vs actual finish for every host |
| 4 | `clean/elo_timeseries.csv` | soccer-elo.csv | Host-nation Elo rating ±5 years around their WC year |
| 5 | `clean/wc_finals.csv` | openfootball + historical records | Top-4 finishers per tournament with host flag |

**Raw data lives in `raw/`. Do not edit raw files — re-run from this notebook instead.**


## Section 0 — Setup

In [22]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

RAW        = Path('raw')
CLEAN      = Path('clean')
OPENFOOTBALL = Path('raw/openfootball_worldcup')

RAW.mkdir(exist_ok=True)
CLEAN.mkdir(exist_ok=True)

print('Directories ready.')
print(f'  raw/   -> {RAW.resolve()}')
print(f'  clean/ -> {CLEAN.resolve()}')


Directories ready.
  raw/   -> /Users/diegomenchaca/Desktop/code/school/DSC106/proj4/raw
  clean/ -> /Users/diegomenchaca/Desktop/code/school/DSC106/proj4/clean


---
## Section 1 — Global Match Results (`Match_Results.csv`)

**Source:** `raw/Match_Results.csv` — 47 399 international matches (1872–2024).

**Key field:** `neutral` (bool) — `False` means the `home_team` is playing on their own soil.
In World Cup matches this cleanly identifies when the **host nation** is playing at home.

**Outputs:** `clean/wc_matches.csv`, `clean/baseline_rates.csv`


### 1.1  Load & inspect

In [23]:
matches = pd.read_csv(RAW / 'Match_Results.csv', parse_dates=['date'])
print(f'Shape : {matches.shape}')
print(f'Cols  : {list(matches.columns)}')
print(f'Dates : {matches["date"].min().date()} → {matches["date"].max().date()}')
matches.head(3)


Shape : (47399, 9)
Cols  : ['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral']
Dates : 1872-11-30 → 2024-07-14


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0,0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4,2,Friendly,London,England,False
2,1874-03-07,Scotland,England,2,1,Friendly,Glasgow,Scotland,False


### 1.2  Quality check

In [24]:
print('=== Null counts ===')
print(matches.isnull().sum())
print()
print('=== neutral distribution ===')
print(matches['neutral'].value_counts())
print()
print('=== Top-10 tournaments ===')
print(matches['tournament'].value_counts().head(10))


=== Null counts ===
date          0
home_team     0
away_team     0
home_score    0
away_score    0
tournament    0
city          0
country       0
neutral       0
dtype: int64

=== neutral distribution ===
neutral
False    34892
True     12507
Name: count, dtype: int64

=== Top-10 tournaments ===
tournament
Friendly                                17995
FIFA World Cup qualification             8169
UEFA Euro qualification                  2824
African Cup of Nations qualification     2124
FIFA World Cup                            964
Copa América                              873
African Cup of Nations                    793
AFC Asian Cup qualification               764
CECAFA Cup                                620
CFU Caribbean Cup qualification           606
Name: count, dtype: int64


### 1.3  Filter to FIFA World Cup

In [25]:
wc = matches[matches['tournament'] == 'FIFA World Cup'].copy()
wc['year'] = wc['date'].dt.year

print(f'WC matches : {len(wc)}')
print(f'Years      : {sorted(wc["year"].unique())}')


WC matches : 964
Years      : [np.int32(1930), np.int32(1934), np.int32(1938), np.int32(1950), np.int32(1954), np.int32(1958), np.int32(1962), np.int32(1966), np.int32(1970), np.int32(1974), np.int32(1978), np.int32(1982), np.int32(1986), np.int32(1990), np.int32(1994), np.int32(1998), np.int32(2002), np.int32(2006), np.int32(2010), np.int32(2014), np.int32(2018), np.int32(2022)]


### 1.4  Add host-team flags and goal differential

In [26]:
# Host country per tournament year.
# 2002 was a dual host: South Korea + Japan (each played on their own soil).
WC_HOSTS = {
    1930: ['Uruguay'],       1934: ['Italy'],          1938: ['France'],
    1950: ['Brazil'],        1954: ['Switzerland'],    1958: ['Sweden'],
    1962: ['Chile'],         1966: ['England'],        1970: ['Mexico'],
    1974: ['Germany'],       1978: ['Argentina'],      1982: ['Spain'],
    1986: ['Mexico'],        1990: ['Italy'],           1994: ['United States'],
    1998: ['France'],        2002: ['South Korea', 'Japan'],
    2006: ['Germany'],       2010: ['South Africa'],   2014: ['Brazil'],
    2018: ['Russia'],        2022: ['Qatar'],
}

# goal_diff is always from home_team's perspective
wc['goal_diff'] = wc['home_score'] - wc['away_score']

def get_host_team(row):
    """neutral=False in a WC row means home_team IS the host playing at home."""
    if not row['neutral']:
        for h in WC_HOSTS.get(row['year'], []):
            if row['home_team'] == h:
                return h
    return None

wc['host_team']     = wc.apply(get_host_team, axis=1)
wc['is_host_match'] = wc['host_team'].notna()

def host_result(row):
    if not row['is_host_match']:
        return None
    gd = row['goal_diff']
    return 'W' if gd > 0 else ('L' if gd < 0 else 'D')

wc['host_result']    = wc.apply(host_result, axis=1)
wc['host_goal_diff'] = wc['goal_diff'].where(wc['is_host_match'])

print(f'Host matches found : {wc["is_host_match"].sum()}')
print()
print('Host match results:')
print(wc[wc['is_host_match']]['host_result'].value_counts())
print()
print('Sample host matches:')
sample_cols = ['year','home_team','away_team','home_score','away_score','host_result','host_goal_diff']
print(wc[wc['is_host_match']][sample_cols].head(10).to_string(index=False))


Host matches found : 121

Host match results:
host_result
W    74
L    25
D    22
Name: count, dtype: int64

Sample host matches:
 year home_team      away_team  home_score  away_score host_result  host_goal_diff
 1930   Uruguay           Peru           1           0           W             1.0
 1930   Uruguay        Romania           4           0           W             4.0
 1930   Uruguay     Yugoslavia           6           1           W             5.0
 1930   Uruguay      Argentina           4           2           W             2.0
 1934     Italy  United States           7           1           W             6.0
 1934     Italy          Spain           1           1           D             0.0
 1934     Italy          Spain           1           0           W             1.0
 1934     Italy        Austria           1           0           W             1.0
 1934     Italy Czechoslovakia           2           1           W             1.0
 1938    France        Belgium          

### 1.5  Export `wc_matches.csv`

In [27]:
out_cols = [
    'date', 'year', 'home_team', 'away_team',
    'home_score', 'away_score', 'goal_diff',
    'city', 'country', 'neutral',
    'is_host_match', 'host_team', 'host_result', 'host_goal_diff',
]
wc[out_cols].to_csv(CLEAN / 'wc_matches.csv', index=False)
print(f'Saved: clean/wc_matches.csv  ({len(wc)} rows x {len(out_cols)} cols)')
wc[out_cols].head(3)


Saved: clean/wc_matches.csv  (964 rows x 14 cols)


,date,year,home_team,away_team,home_score,away_score,goal_diff,city,country,neutral,is_host_match,host_team,host_result,host_goal_diff
1444,1930-07-13,1930,Belgium,United States,0,3,-3,Montevideo,Uruguay,True,False,None,None,NaN
1445,1930-07-13,1930,France,Mexico,4,1,3,Montevideo,Uruguay,True,False,None,None,NaN
1446,1930-07-14,1930,Brazil,Yugoslavia,1,2,-1,Montevideo,Uruguay,True,False,None,None,NaN


### 1.6  Compute baseline win rates

In [28]:
# Classify every international match result
all_m = matches.dropna(subset=['home_score', 'away_score']).copy()

def classify(row):
    if row['home_score'] > row['away_score']: return 'home_win'
    if row['home_score'] < row['away_score']: return 'away_win'
    return 'draw'

all_m['result'] = all_m.apply(classify, axis=1)

# Three venue contexts
home_v    = all_m[all_m['neutral'] == False]['result'].value_counts(normalize=True)
neutral_v = all_m[all_m['neutral'] == True ]['result'].value_counts(normalize=True)

wc_host_m = wc[wc['is_host_match']].copy()
wc_host_m['result'] = wc_host_m['host_result'].map({'W': 'home_win', 'L': 'away_win', 'D': 'draw'})
wc_host_v = wc_host_m['result'].value_counts(normalize=True)

rates = (
    pd.DataFrame({'home_venue': home_v, 'neutral_venue': neutral_v, 'wc_host_match': wc_host_v})
    .T
    .rename_axis('venue_type')
    .reset_index()
)
for col in ['home_win', 'draw', 'away_win']:
    if col not in rates.columns:
        rates[col] = 0.0

rates = rates[['venue_type', 'home_win', 'draw', 'away_win']].round(4)
print(rates.to_string(index=False))


   venue_type  home_win   draw  away_win
   home_venue    0.5074 0.2287    0.2639
neutral_venue    0.4426 0.2240    0.3335
wc_host_match    0.6116 0.1818    0.2066


### 1.7  Export `baseline_rates.csv`

In [29]:
rates.to_csv(CLEAN / 'baseline_rates.csv', index=False)
print(f'Saved: clean/baseline_rates.csv  ({len(rates)} rows)')


Saved: clean/baseline_rates.csv  (3 rows)


---
## Section 2 — Soccer Elo Ratings (`soccer-elo.csv`)

**Source:** `raw/soccer-elo.csv` — annual Elo snapshots (1901–2023), ~17 k rows.

**Known issue:** the `one_year_change_*` columns use the Unicode minus sign `−` (U+2212)
instead of the ASCII hyphen-minus `-`. This prevents `pd.read_csv` from parsing them
as numbers. We fix this before coercing to float.

**Strategy for "expected finish":**
1. For each host-year, collect all WC participants from `wc_matches.csv`.
2. Pull each participant's Elo rating from `year − 1` (pre-tournament snapshot).
3. Rank the host among participants by Elo → `elo_rank_among_participants`.
4. Compare to `actual_place` (hardcoded from historical records).
5. `delta = elo_rank_among_participants − actual_place`
   → positive = outperformed expectations; negative = underperformed.

**Outputs:** `clean/host_elo_delta.csv`, `clean/elo_timeseries.csv`


### 2.1  Load & inspect

In [30]:
elo_raw = pd.read_csv(RAW / 'soccer-elo.csv', dtype=str)
print(f'Shape : {elo_raw.shape}')
print(f'Cols  : {list(elo_raw.columns)}')
print()
# Show the encoding issue in the change columns
print('Sample of one_year_change_rating (raw):')
print(elo_raw['one_year_change_rating'].head(6).tolist())


Shape : (17200, 17)
Cols  : ['year', 'rank', 'team', 'rating', 'average_rank', 'average_rating', 'one_year_change_rank', 'one_year_change_rating', 'matches_total', 'matches_home', 'matches_away', 'matches_neutral', 'matches_wins', 'matches_losses', 'matches_draws', 'goals_for', 'goals_against']

Sample of one_year_change_rating (raw):
['−6', '−6', '35', '−23', '−', '−18']


### 2.2  Clean and fix encoding

In [31]:
UNICODE_MINUS = chr(8722)  # U+2212 '−'

def fix_num(series):
    """Replace Unicode minus with ASCII hyphen, then coerce to float."""
    return pd.to_numeric(
        series.astype(str).str.replace(UNICODE_MINUS, '-', regex=False).str.strip(),
        errors='coerce'
    )

elo = pd.DataFrame({
    'year'                  : fix_num(elo_raw['year']),
    'rank'                  : fix_num(elo_raw['rank']),
    'team'                  : elo_raw['team'].str.strip(),
    'rating'                : fix_num(elo_raw['rating']),
    'one_year_change_rank'  : fix_num(elo_raw['one_year_change_rank']),
    'one_year_change_rating': fix_num(elo_raw['one_year_change_rating']),
})

# Drop the repeated header row that sometimes appears at the end of the file
elo = elo.dropna(subset=['year', 'rank']).copy()
elo[['year', 'rank']] = elo[['year', 'rank']].astype(int)

print(f'Clean shape : {elo.shape}')
print(f'Year range  : {elo["year"].min()} – {elo["year"].max()}')
print(f'Unique teams: {elo["team"].nunique()}')
print()
print(elo.head(4))


Clean shape : (17200, 6)
Year range  : 1901 – 2023
Unique teams: 290

   year  rank      team  rating  one_year_change_rank  one_year_change_rating
0  1901     1   England    2013                   0.0                    -6.0
1  1901     2  Scotland    1973                   0.0                    -6.0
2  1901     3     Wales    1476                   0.0                    35.0
3  1901     4   Ireland    1338                   0.0                   -23.0


### 2.3  Hardcoded host actual-finish lookup

In [32]:
# actual_place : ordinal finish (1 = champion)
# For tied elimination positions, use the midpoint of the tied range:
#   QF exit in a 32-team field = tied 5th–8th  → use 6
#   R16 exit in a 32-team field = tied 9th–16th → use 12
#   GS exit in a 32-team field  = tied 17th–32nd → use 25
#
# Notable edge cases:
#   1950 Brazil: lost the deciding Final Pool match (no formal Final) → 2nd
#   1982 Spain : eliminated in the second-round group stage (24-team format) → 12th
#   2010 South Africa: first host ever to exit in the group stage → 25th
#   2022 Qatar        : second host to exit in group stage → 25th

HOST_FINISHES = {
    (1930, 'Uruguay')       : {'actual_place':  1, 'round_reached': 'Champion',     'n_teams': 13},
    (1934, 'Italy')         : {'actual_place':  1, 'round_reached': 'Champion',     'n_teams': 16},
    (1938, 'France')        : {'actual_place':  8, 'round_reached': 'Quarterfinal', 'n_teams': 16},
    (1950, 'Brazil')        : {'actual_place':  2, 'round_reached': 'Final',        'n_teams': 13},
    (1954, 'Switzerland')   : {'actual_place':  8, 'round_reached': 'Quarterfinal', 'n_teams': 16},
    (1958, 'Sweden')        : {'actual_place':  2, 'round_reached': 'Final',        'n_teams': 16},
    (1962, 'Chile')         : {'actual_place':  3, 'round_reached': 'Third Place',  'n_teams': 16},
    (1966, 'England')       : {'actual_place':  1, 'round_reached': 'Champion',     'n_teams': 16},
    (1970, 'Mexico')        : {'actual_place':  8, 'round_reached': 'Quarterfinal', 'n_teams': 16},
    (1974, 'Germany')       : {'actual_place':  1, 'round_reached': 'Champion',     'n_teams': 16},
    (1978, 'Argentina')     : {'actual_place':  1, 'round_reached': 'Champion',     'n_teams': 16},
    (1982, 'Spain')         : {'actual_place': 12, 'round_reached': 'Second Round', 'n_teams': 24},
    (1986, 'Mexico')        : {'actual_place':  8, 'round_reached': 'Quarterfinal', 'n_teams': 24},
    (1990, 'Italy')         : {'actual_place':  3, 'round_reached': 'Third Place',  'n_teams': 24},
    (1994, 'United States') : {'actual_place': 16, 'round_reached': 'Round of 16',  'n_teams': 24},
    (1998, 'France')        : {'actual_place':  1, 'round_reached': 'Champion',     'n_teams': 32},
    (2002, 'South Korea')   : {'actual_place':  4, 'round_reached': 'Fourth Place', 'n_teams': 32},
    (2002, 'Japan')         : {'actual_place': 16, 'round_reached': 'Round of 16',  'n_teams': 32},
    (2006, 'Germany')       : {'actual_place':  3, 'round_reached': 'Third Place',  'n_teams': 32},
    (2010, 'South Africa')  : {'actual_place': 25, 'round_reached': 'Group Stage',  'n_teams': 32},
    (2014, 'Brazil')        : {'actual_place':  4, 'round_reached': 'Fourth Place', 'n_teams': 32},
    (2018, 'Russia')        : {'actual_place':  8, 'round_reached': 'Quarterfinal', 'n_teams': 32},
    (2022, 'Qatar')         : {'actual_place': 25, 'round_reached': 'Group Stage',  'n_teams': 32},
}

preview = pd.DataFrame([{'year': k[0], 'host': k[1], **v} for k, v in HOST_FINISHES.items()])
print(f'{len(HOST_FINISHES)} host-year entries:')
print(preview.to_string(index=False))


23 host-year entries:
 year          host  actual_place round_reached  n_teams
 1930       Uruguay             1      Champion       13
 1934         Italy             1      Champion       16
 1938        France             8  Quarterfinal       16
 1950        Brazil             2         Final       13
 1954   Switzerland             8  Quarterfinal       16
 1958        Sweden             2         Final       16
 1962         Chile             3   Third Place       16
 1966       England             1      Champion       16
 1970        Mexico             8  Quarterfinal       16
 1974       Germany             1      Champion       16
 1978     Argentina             1      Champion       16
 1982         Spain            12  Second Round       24
 1986        Mexico             8  Quarterfinal       24
 1990         Italy             3   Third Place       24
 1994 United States            16   Round of 16       24
 1998        France             1      Champion       32
 2002   S

### 2.4  Derive WC participants from match data

In [33]:
# One edge case: soccer-elo uses 'West Germany' for pre-1990 entries,
# but Match_Results uses 'Germany' throughout. We bridge this only for the
# 1974 WC host lookup (the only pre-1990 host named 'Germany' in our data).

def elo_lookup_name(year, team):
    """Map Match_Results team name to soccer-elo team name."""
    if team == 'Germany' and year < 1990:
        return 'West Germany'
    return team

# Collect all unique teams per WC year
wc_participants = {}
for year, grp in wc.groupby('year'):
    wc_participants[year] = set(grp['home_team'].tolist() + grp['away_team'].tolist())

print('Team counts per tournament year:')
for yr in sorted(wc_participants):
    print(f'  {yr}: {len(wc_participants[yr])} teams')


Team counts per tournament year:
  1930: 13 teams
  1934: 16 teams
  1938: 15 teams
  1950: 13 teams
  1954: 16 teams
  1958: 16 teams
  1962: 16 teams
  1966: 16 teams
  1970: 16 teams
  1974: 16 teams
  1978: 16 teams
  1982: 24 teams
  1986: 24 teams
  1990: 24 teams
  1994: 24 teams
  1998: 32 teams
  2002: 32 teams
  2006: 32 teams
  2010: 32 teams
  2014: 32 teams
  2018: 32 teams
  2022: 32 teams


### 2.5  Compute Elo-based expected finish and delta

In [34]:
rows = []

for (year, host), info in HOST_FINISHES.items():
    elo_year  = year - 1          # pre-tournament snapshot
    elo_name  = elo_lookup_name(year, host)
    participants = wc_participants.get(year, set())

    # Elo data for the lookup year
    elo_yr = elo[elo['year'] == elo_year]
    elo_idx = elo_yr.set_index('team')

    # Host's global Elo rank and rating
    if elo_name in elo_idx.index:
        host_rating      = float(elo_idx.loc[elo_name, 'rating'])
        host_global_rank = int(elo_idx.loc[elo_name, 'rank'])
    else:
        host_rating, host_global_rank = np.nan, np.nan

    # Rank host among tournament participants
    participant_ratings = {}
    for team in participants:
        lookup = elo_lookup_name(year, team)
        if lookup in elo_idx.index:
            participant_ratings[lookup] = float(elo_idx.loc[lookup, 'rating'])

    ranked = sorted(participant_ratings.items(), key=lambda x: x[1], reverse=True)
    elo_rank_among = next(
        (i + 1 for i, (t, _) in enumerate(ranked) if t == elo_name),
        np.nan,
    )

    delta = (elo_rank_among - info['actual_place']) if not np.isnan(elo_rank_among) else np.nan

    rows.append({
        'year'                        : year,
        'host'                        : host,
        'pre_tournament_elo'          : round(host_rating, 1) if not np.isnan(host_rating) else np.nan,
        'elo_global_rank'             : host_global_rank,
        'elo_rank_among_participants' : int(elo_rank_among) if not np.isnan(elo_rank_among) else np.nan,
        'n_participants_rated'        : len(ranked),
        'actual_place'                : info['actual_place'],
        'round_reached'               : info['round_reached'],
        'n_teams'                     : info['n_teams'],
        'delta'                       : round(float(delta), 2) if not np.isnan(delta) else np.nan,
    })

host_elo_df = pd.DataFrame(rows)
print(host_elo_df.to_string(index=False))


 year          host  pre_tournament_elo  elo_global_rank  elo_rank_among_participants  n_participants_rated  actual_place round_reached  n_teams  delta
 1930       Uruguay              1963.0                3                            2                    13             1      Champion       13    1.0
 1934         Italy              2012.0                2                            2                    15             1      Champion       16    1.0
 1938        France              1618.0               30                            9                    13             8  Quarterfinal       16    1.0
 1950        Brazil              1986.0                4                            3                    13             2         Final       13    1.0
 1954   Switzerland              1669.0               27                           14                    16             8  Quarterfinal       16    6.0
 1958        Sweden              1797.0               17                           11   

### 2.6  Export `host_elo_delta.csv`

In [35]:
host_elo_df.to_csv(CLEAN / 'host_elo_delta.csv', index=False)
print(f'Saved: clean/host_elo_delta.csv  ({len(host_elo_df)} rows)')


Saved: clean/host_elo_delta.csv  (23 rows)


### 2.7  Build Elo time-series for host nations

In [36]:
# For each host, pull their Elo rating from (wc_year - 5) to (wc_year + 5).
# This powers the 'Elo trajectory around hosting' line-chart in proposal_viz.
WINDOW = 5

ts_rows = []
for (year, host), info in HOST_FINISHES.items():
    elo_name = elo_lookup_name(year, host)
    for y in range(year - WINDOW, year + WINDOW + 1):
        yr_slice = elo[(elo['year'] == y) & (elo['team'] == elo_name)]
        if len(yr_slice) == 0:
            continue
        ts_rows.append({
            'wc_year'       : year,
            'host'          : host,
            'year'          : y,
            'years_from_wc' : y - year,
            'elo_rating'    : float(yr_slice['rating'].iloc[0]),
            'elo_rank'      : int(yr_slice['rank'].iloc[0]),
        })

elo_ts = pd.DataFrame(ts_rows)
print(f'Shape: {elo_ts.shape}')
print()
print('Brazil 2014 trajectory:')
print(elo_ts[(elo_ts['host'] == 'Brazil') & (elo_ts['wc_year'] == 2014)].to_string(index=False))


Shape: (249, 6)

Brazil 2014 trajectory:
 wc_year   host  year  years_from_wc  elo_rating  elo_rank
    2014 Brazil  2009             -5      2103.0         1
    2014 Brazil  2010             -4      2095.0         3
    2014 Brazil  2011             -3      2064.0         4
    2014 Brazil  2012             -2      2060.0         2
    2014 Brazil  2013             -1      2132.0         1
    2014 Brazil  2014              0      2042.0         3
    2014 Brazil  2015              1      2038.0         2
    2014 Brazil  2016              2      2087.0         1
    2014 Brazil  2017              3      2113.0         1
    2014 Brazil  2018              4      2136.0         1
    2014 Brazil  2019              5      2082.0         2


### 2.8  Export `elo_timeseries.csv`

In [37]:
elo_ts.to_csv(CLEAN / 'elo_timeseries.csv', index=False)
print(f'Saved: clean/elo_timeseries.csv  ({len(elo_ts)} rows)')


Saved: clean/elo_timeseries.csv  (249 rows)


---
## Section 3 — OpenFootball World Cup Data (`raw/openfootball_worldcup/`)

**Source:** `raw/openfootball_worldcup/<year>--<host>/cup.txt` — one file per tournament.

**Format:** Custom plaintext. Section headers begin with `▪` (single bullet = round header,
double `▪▪` = group sub-header). Matches appear as free-text lines within each section.

**Limitation discovered:** The `cup.txt` files for 2002, 2018, and 2022 contain only the
group-stage matches; knockout-round data is absent. For those years the host-nation finish
comes from the `HOST_FINISHES` lookup in Section 2. Attendance data is not present in any
`cup.txt` file and will require a supplementary dataset for the final D3 interactive.

**Output:** `clean/wc_finals.csv` — one row per tournament, top-4 finishers + host info.


### 3.1  Hardcoded tournament top-4 records

In [38]:
# Top-4 finishers per tournament, sourced from FIFA official records.
# Used to power the tournament-results heatmap in proposal_viz.
WC_TOP4 = [
    # year  champion           runner_up           third               fourth
    (1930,  'Uruguay',         'Argentina',        'United States',    'Yugoslavia'),
    (1934,  'Italy',           'Czechoslovakia',   'Germany',          'Austria'),
    (1938,  'Italy',           'Hungary',          'Brazil',           'Sweden'),
    (1950,  'Uruguay',         'Brazil',           'Sweden',           'Spain'),
    (1954,  'Germany',         'Hungary',          'Austria',          'Uruguay'),
    (1958,  'Brazil',          'Sweden',           'France',           'Germany'),
    (1962,  'Brazil',          'Czechoslovakia',   'Chile',            'Yugoslavia'),
    (1966,  'England',         'Germany',          'Portugal',         'Russia'),
    (1970,  'Brazil',          'Italy',            'Germany',          'Uruguay'),
    (1974,  'Germany',         'Netherlands',      'Poland',           'Brazil'),
    (1978,  'Argentina',       'Netherlands',      'Brazil',           'Italy'),
    (1982,  'Italy',           'Germany',          'Poland',           'France'),
    (1986,  'Argentina',       'Germany',          'France',           'Belgium'),
    (1990,  'Germany',         'Argentina',        'Italy',            'England'),
    (1994,  'Brazil',          'Italy',            'Sweden',           'Bulgaria'),
    (1998,  'France',          'Brazil',           'Croatia',          'Netherlands'),
    (2002,  'Brazil',          'Germany',          'Turkey',           'South Korea'),
    (2006,  'Italy',           'France',           'Germany',          'Portugal'),
    (2010,  'Spain',           'Netherlands',      'Germany',          'Uruguay'),
    (2014,  'Germany',         'Argentina',        'Netherlands',      'Brazil'),
    (2018,  'France',          'Croatia',          'Belgium',          'England'),
    (2022,  'Argentina',       'France',           'Croatia',          'Morocco'),
]

wc_top4 = pd.DataFrame(WC_TOP4, columns=['year','champion','runner_up','third','fourth'])
print(wc_top4.to_string(index=False))


 year  champion      runner_up         third      fourth
 1930   Uruguay      Argentina United States  Yugoslavia
 1934     Italy Czechoslovakia       Germany     Austria
 1938     Italy        Hungary        Brazil      Sweden
 1950   Uruguay         Brazil        Sweden       Spain
 1954   Germany        Hungary       Austria     Uruguay
 1958    Brazil         Sweden        France     Germany
 1962    Brazil Czechoslovakia         Chile  Yugoslavia
 1966   England        Germany      Portugal      Russia
 1970    Brazil          Italy       Germany     Uruguay
 1974   Germany    Netherlands        Poland      Brazil
 1978 Argentina    Netherlands        Brazil       Italy
 1982     Italy        Germany        Poland      France
 1986 Argentina        Germany        France     Belgium
 1990   Germany      Argentina         Italy     England
 1994    Brazil          Italy        Sweden    Bulgaria
 1998    France         Brazil       Croatia Netherlands
 2002    Brazil        Germany 

### 3.2  Attach host metadata and parse cup.txt where available

In [39]:
# Round-section classifier for cup.txt ▪ headers
ROUND_PRIORITY = {
    'Group Stage' : 1,
    'Round of 16' : 2,
    'Quarterfinal': 3,
    'Semifinal'   : 4,
    'Third Place' : 5,
    'Final'       : 6,
}

def classify_header(text):
    t = text.lower()
    if 'group' in t and 'second' not in t:
        return 'Group Stage'
    if any(k in t for k in ['round of 16', 'second round', 'round 2', 'eighth']):
        return 'Round of 16'
    if 'quarter' in t:
        return 'Quarterfinal'
    if 'semi' in t:
        return 'Semifinal'
    if 'third' in t or '3rd' in t or 'play-off' in t or 'playoff' in t:
        return 'Third Place'
    if t.rstrip().endswith('final') or t.strip() == 'final':
        return 'Final'
    return None

def split_into_sections(content):
    """Split cup.txt content into (header, body) pairs on single-▪ lines."""
    sections = []
    current_header = None
    current_body   = []
    for line in content.splitlines():
        stripped = line.strip()
        # Single ▪ (not ▪▪) marks a new round section
        if stripped.startswith('▪') and not stripped.startswith('▪▪'):
            if current_header is not None:
                sections.append((current_header, '\n'.join(current_body)))
            current_header = stripped.lstrip('▪').strip()
            current_body   = []
        elif current_header is not None:
            current_body.append(line)
    if current_header is not None:
        sections.append((current_header, '\n'.join(current_body)))
    return sections

# cup.txt team-name variants → Match_Results.csv names
CUP_NORM = {
    'usa'             : 'United States',
    'west germany'    : 'Germany',
    'ivory coast'     : 'Côte d\'Ivoire',
    'soviet union'    : 'Russia',
}

def normalise_cup_name(name):
    return CUP_NORM.get(name.lower(), name)

def parse_cup_txt(filepath, known_teams):
    """Return {team: deepest_round_label} for all known_teams found in the file."""
    try:
        content = filepath.read_text(encoding='utf-8', errors='replace')
    except Exception as e:
        print(f'  Could not read {filepath}: {e}')
        return {}

    sections = split_into_sections(content)
    team_depth = {}

    for header, body in sections:
        round_label = classify_header(header)
        if round_label is None:
            continue
        round_num = ROUND_PRIORITY[round_label]

        for team in known_teams:
            aliases = {team, normalise_cup_name(team)}
            found = any(
                re.search(r'\b' + re.escape(alias) + r'\b', body, re.IGNORECASE)
                for alias in aliases
            )
            if found:
                current = ROUND_PRIORITY.get(team_depth.get(team, 'Group Stage'), 1)
                if round_num > current:
                    team_depth[team] = round_label

    # Teams not found in any section default to Group Stage
    for team in known_teams:
        team_depth.setdefault(team, 'Group Stage')

    return team_depth

# Locate cup.txt files
year_to_folder = {}
if OPENFOOTBALL.exists():
    for folder in OPENFOOTBALL.iterdir():
        if folder.is_dir() and folder.name[0].isdigit():
            yr = int(folder.name.split('--')[0])
            year_to_folder[yr] = folder

# Parse and record results
tournament_results = {}
print('Parsing cup.txt files:')
for year in sorted(year_to_folder):
    cup_txt = year_to_folder[year] / 'cup.txt'
    if not cup_txt.exists():
        print(f'  {year}: file missing — skipping')
        continue
    participants = wc_participants.get(year, set())
    team_rounds  = parse_cup_txt(cup_txt, participants)
    tournament_results[year] = team_rounds
    # Detect how many rounds were found (beyond Group Stage)
    rounds_found = set(v for v in team_rounds.values() if v != 'Group Stage')
    print(f'  {year}: {len(participants)} teams | extra rounds found: {rounds_found or "none (group-stage only)"}')


Parsing cup.txt files:
  1930: 13 teams | extra rounds found: {'Semifinal', 'Final'}
  1934: 16 teams | extra rounds found: {'Quarterfinal', 'Final', 'Third Place'}
  1938: 15 teams | extra rounds found: {'Quarterfinal', 'Final', 'Third Place'}
  1950: 13 teams | extra rounds found: none (group-stage only)
  1954: 16 teams | extra rounds found: {'Quarterfinal', 'Final', 'Third Place'}
  1958: 16 teams | extra rounds found: {'Quarterfinal', 'Final', 'Third Place'}
  1962: 16 teams | extra rounds found: {'Quarterfinal', 'Final', 'Third Place'}
  1966: 16 teams | extra rounds found: {'Quarterfinal', 'Final', 'Third Place'}
  1970: 16 teams | extra rounds found: {'Quarterfinal', 'Final', 'Third Place'}
  1974: 16 teams | extra rounds found: {'Final', 'Third Place'}
  1978: 16 teams | extra rounds found: {'Final', 'Third Place'}
  1982: 24 teams | extra rounds found: {'Final', 'Third Place'}
  1986: 24 teams | extra rounds found: none (group-stage only)
  1990: 24 teams | extra rounds found

### 3.3  Build `wc_finals.csv`

In [40]:
# Merge top-4 table with host info and host finish from HOST_FINISHES
host_lookup = {}
for (year, host), info in HOST_FINISHES.items():
    if year not in host_lookup:
        host_lookup[year] = []
    host_lookup[year].append({'host': host, **info})

finals_rows = []
for rec in WC_TOP4:
    year = rec[0]
    hosts_this_year = host_lookup.get(year, [])
    # Primary host (first entry; 2002 has two)
    primary = hosts_this_year[0] if hosts_this_year else {}
    secondary = hosts_this_year[1] if len(hosts_this_year) > 1 else {}

    finals_rows.append({
        'year'               : year,
        'champion'           : rec[1],
        'runner_up'          : rec[2],
        'third'              : rec[3],
        'fourth'             : rec[4],
        'host'               : primary.get('host', ''),
        'host_actual_place'  : primary.get('actual_place', np.nan),
        'host_round_reached' : primary.get('round_reached', ''),
        'host_n_teams'       : primary.get('n_teams', np.nan),
        'host2'              : secondary.get('host', ''),
        'host2_actual_place' : secondary.get('actual_place', np.nan),
        'host2_round_reached': secondary.get('round_reached', ''),
        'host_won'           : rec[1] in [primary.get('host',''), secondary.get('host','')],
    })

wc_finals = pd.DataFrame(finals_rows)
print(wc_finals.to_string(index=False))


 year  champion      runner_up         third      fourth          host  host_actual_place host_round_reached  host_n_teams host2  host2_actual_place host2_round_reached  host_won
 1930   Uruguay      Argentina United States  Yugoslavia       Uruguay                  1           Champion            13                       NaN                          True
 1934     Italy Czechoslovakia       Germany     Austria         Italy                  1           Champion            16                       NaN                          True
 1938     Italy        Hungary        Brazil      Sweden        France                  8       Quarterfinal            16                       NaN                         False
 1950   Uruguay         Brazil        Sweden       Spain        Brazil                  2              Final            13                       NaN                         False
 1954   Germany        Hungary       Austria     Uruguay   Switzerland                  8       Quarterfi

### 3.4  Export `wc_finals.csv`

In [41]:
wc_finals.to_csv(CLEAN / 'wc_finals.csv', index=False)
print(f'Saved: clean/wc_finals.csv  ({len(wc_finals)} rows)')


Saved: clean/wc_finals.csv  (22 rows)


---
## Section 4 — Dataset Summary for `proposal_viz.ipynb`

All five clean files are now ready in `clean/`. Import them in `proposal_viz.ipynb` with:

```python
import pandas as pd
from pathlib import Path
CLEAN = Path('clean')

wc_matches    = pd.read_csv(CLEAN / 'wc_matches.csv',    parse_dates=['date'])
baseline      = pd.read_csv(CLEAN / 'baseline_rates.csv')
host_elo      = pd.read_csv(CLEAN / 'host_elo_delta.csv')
elo_ts        = pd.read_csv(CLEAN / 'elo_timeseries.csv')
wc_finals     = pd.read_csv(CLEAN / 'wc_finals.csv')
```

### Suggested proposal visualisations

| # | Chart type | Data file | X / Y axes |
|---|-----------|-----------|-----------|
| 1 | Slopegraph | `host_elo_delta.csv` | `elo_rank_among_participants` vs `actual_place` |
| 2 | Grouped bar | `baseline_rates.csv` | Venue type vs win/draw/loss rates |
| 3 | Multi-line | `elo_timeseries.csv` | `years_from_wc` vs `elo_rating`, one line per host |
| 4 | KDE / histogram | `wc_matches.csv` | `host_goal_diff` vs all other `goal_diff` |
| 5 | Scatter | `host_elo_delta.csv` | `year` vs `delta`, sized by `pre_tournament_elo` |
| 6 | Dot / strip | `host_elo_delta.csv` | `round_reached` distribution across all 23 host entries |

### Attendance data note

No per-match attendance figures are present in the OpenFootball `cup.txt` files.
The `rsssf/` sub-directory contains `Att:` entries only for select finals.
For the final D3 interactive (Section 2 scatterplot), supplement with the
[Kaggle World Cup Matches dataset](https://www.kaggle.com/datasets/abecklas/fifa-world-cup)
which includes per-match attendance from 1930–2018.


In [42]:
# Quick health check on all clean files
for fname in ['wc_matches.csv', 'baseline_rates.csv',
              'host_elo_delta.csv', 'elo_timeseries.csv', 'wc_finals.csv']:
    path = CLEAN / fname
    if path.exists():
        df = pd.read_csv(path)
        print(f'{fname:30s}  {df.shape[0]:>5} rows  x  {df.shape[1]:>2} cols  |  columns: {list(df.columns)}')
    else:
        print(f'{fname:30s}  *** NOT FOUND ***')


wc_matches.csv                    964 rows  x  14 cols  |  columns: ['date', 'year', 'home_team', 'away_team', 'home_score', 'away_score', 'goal_diff', 'city', 'country', 'neutral', 'is_host_match', 'host_team', 'host_result', 'host_goal_diff']
baseline_rates.csv                  3 rows  x   4 cols  |  columns: ['venue_type', 'home_win', 'draw', 'away_win']
host_elo_delta.csv                 23 rows  x  10 cols  |  columns: ['year', 'host', 'pre_tournament_elo', 'elo_global_rank', 'elo_rank_among_participants', 'n_participants_rated', 'actual_place', 'round_reached', 'n_teams', 'delta']
elo_timeseries.csv                249 rows  x   6 cols  |  columns: ['wc_year', 'host', 'year', 'years_from_wc', 'elo_rating', 'elo_rank']
wc_finals.csv                      22 rows  x  13 cols  |  columns: ['year', 'champion', 'runner_up', 'third', 'fourth', 'host', 'host_actual_place', 'host_round_reached', 'host_n_teams', 'host2', 'host2_actual_place', 'host2_round_reached', 'host_won']
